In [1]:
from __future__ import annotations
from dataclasses import dataclass
from typing import Iterable, Iterator, Optional, cast

import torch

from core.events import EventBatch
from data.interfaces import DataSpec, EventStreamDataset


@dataclass
class SpringMassConfig:
    name: str = "spring_mass"

    # number of nodes ("masses") in a line: 0--1--2--3--...
    num_nodes: int = 20

    # how many time bins to simulate
    num_bins: int = 200

    # simple dynamics params
    dt: float = 0.05
    spring_k: float = 1.0
    damping: float = 0.98

    # rest spacing between neighbors in equilibrium
    rest_length: float = 1.0

    # random initial wiggle size
    init_pos_noise: float = 0.25
    init_vel_noise: float = 0.05

    # only emit an event if interaction magnitude is above this
    force_threshold: float = 0.10

    # optional device for yielded batches
    device: Optional[torch.device] = None


class SpringMassDataset(EventStreamDataset):
    def __init__(self, cfg: SpringMassConfig):
        self.cfg = cfg

        # feature vector:
        # [dx, dv, extension, force]
        self._event_dim = 4

    def spec(self) -> DataSpec:
        # num_events is approximate here, since thresholding means
        # the exact number depends on the simulation.
        approx_events = self.cfg.num_bins * max(1, self.cfg.num_nodes - 1)

        return DataSpec(
            name=self.cfg.name,
            num_nodes=self.cfg.num_nodes,
            event_dim=self._event_dim,
            num_bins=self.cfg.num_bins,
            num_events=approx_events,
            extra={
                "dt": self.cfg.dt,
                "spring_k": self.cfg.spring_k,
                "damping": self.cfg.damping,
                "rest_length": self.cfg.rest_length,
                "force_threshold": self.cfg.force_threshold,
            },
        )

    def bins(self, split: str = "train") -> Iterable[EventBatch]:
        # for now, ignore split just like toy.py
        return _SpringMassStream(self.cfg, self._event_dim)


@dataclass
class _SpringMassStream(Iterable[EventBatch]):
    cfg: SpringMassConfig
    event_dim: int

    def __iter__(self) -> Iterator[EventBatch]:
        device = self.cfg.device
        N = self.cfg.num_nodes
        dt = self.cfg.dt
        k = self.cfg.spring_k
        damping = self.cfg.damping
        rest = self.cfg.rest_length
        thr = self.cfg.force_threshold

        # positions start near equally spaced line:
        # [0, 1, 2, 3, ...] plus a little noise
        x = torch.arange(N, dtype=torch.float32) * rest
        x = x + self.cfg.init_pos_noise * torch.randn(N, dtype=torch.float32)

        # small random initial velocities
        v = self.cfg.init_vel_noise * torch.randn(N, dtype=torch.float32)

        for b in range(self.cfg.num_bins):
            src_list = []
            dst_list = []
            feat_list = []

            # net force on each node
            net_force = torch.zeros(N, dtype=torch.float32)

            # only adjacent neighbors are connected:
            # (0,1), (1,2), ..., (N-2, N-1)
            for i in range(N - 1):
                j = i + 1

                dx = x[j] - x[i]              # current spacing
                dv = v[j] - v[i]              # relative velocity
                extension = dx - rest         # how much spring is stretched/squished
                force = k * extension         # simple Hooke-ish force magnitude

                # apply equal/opposite forces to the two nodes
                net_force[i] += force
                net_force[j] -= force

                # threshold: only record "interesting enough" interactions
                if torch.abs(force) > thr:
                    src_list.append(i)
                    dst_list.append(j)
                    feat_list.append([
                        float(dx),
                        float(dv),
                        float(extension),
                        float(force),
                    ])

            # if no interactions passed threshold, emit an empty batch
            if len(src_list) == 0:
                src = torch.empty(0, dtype=torch.long)
                dst = torch.empty(0, dtype=torch.long)
                t = torch.empty(0, dtype=torch.long)
                feats = torch.empty((0, self.event_dim), dtype=torch.float32)
            else:
                src = torch.tensor(src_list, dtype=torch.long)
                dst = torch.tensor(dst_list, dtype=torch.long)
                t = torch.full((len(src_list),), b, dtype=torch.long)
                feats = torch.tensor(feat_list, dtype=torch.float32)

            eb = EventBatch(
                src=cast(torch.LongTensor, src),
                dst=cast(torch.LongTensor, dst),
                t=cast(torch.LongTensor, t),
                features=feats,
            )

            if device is not None:
                eb = eb.to(device)

            yield eb

            # --- update state for next timestep ---
            # super simple dynamics:
            # acceleration = force   (assume mass = 1)
            a = net_force

            # update velocity and damp it a little
            v = damping * (v + dt * a)

            # update position
            x = x + dt * v

In [4]:
# from data.spring_mass import SpringMassConfig, SpringMassDataset

ds = SpringMassDataset(SpringMassConfig(
    num_nodes=8,
    num_bins=10,
    force_threshold=0.10,
))

print(ds.spec())

for b, eb in enumerate(ds.bins()):
    print(f"bin {b}: {eb.src.numel()} events")
    print("src:", eb.src[:5])
    print("dst:", eb.dst[:5])
    print("features:", eb.features[:5] if eb.features is not None else None)
    print()

DataSpec(name='spring_mass', num_nodes=8, event_dim=4, num_events=70, num_bins=10, extra={'dt': 0.05, 'spring_k': 1.0, 'damping': 0.98, 'rest_length': 1.0, 'force_threshold': 0.1})
bin 0: 5 events
src: tensor([1, 2, 3, 5, 6])
dst: tensor([2, 3, 4, 6, 7])
features: tensor([[ 0.8111,  0.0053, -0.1889, -0.1889],
        [ 1.1827,  0.1010,  0.1827,  0.1827],
        [ 1.2084,  0.0065,  0.2084,  0.2084],
        [ 0.8467, -0.0379, -0.1533, -0.1533],
        [ 1.1038,  0.0307,  0.1038,  0.1038]])

bin 1: 5 events
src: tensor([1, 2, 3, 5, 6])
dst: tensor([2, 3, 4, 6, 7])
features: tensor([[ 0.8127,  0.0320, -0.1873, -0.1873],
        [ 1.1868,  0.0820,  0.1868,  0.1868],
        [ 1.2081, -0.0072,  0.2081,  0.2081],
        [ 0.8457, -0.0191, -0.1543, -0.1543],
        [ 1.1044,  0.0124,  0.1044,  0.1044]])

bin 2: 5 events
src: tensor([1, 2, 3, 5, 6])
dst: tensor([2, 3, 4, 6, 7])
features: tensor([[ 8.1561e-01,  5.8252e-02, -1.8439e-01, -1.8439e-01],
        [ 1.1899e+00,  6.3066e-02,  1.899